# [S3L0 demo] Write a new Jupyter notebook

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-643

See the associated:

  * Python module: [S3L0_demo_processor.py](./S3L0_demo_processor.py)
  * YAML file: [S3L0_demo_processor.yaml](./S3L0_demo_processor.yaml)

## 1. Initialisation

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
init_dask_cluster_eopf(scale=2)
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)
display(dask_cluster_staging)

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.1"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/7d8efdd716dd462e8b48d19f254858da/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| lz4     | 4.4.3  | 4.4.4     | None    |
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf' are up: 0/2
Dask workers for 'dask-eopf' are up: 2/2
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
Create new dask cluster
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/ce7f4288e6be44a0a1f38f25cecad98f/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| lz4     | 4.4.3  | 4.4.4     | None    |
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-staging' are up: 0/2
Dask workers for 'dask-staging' are up: 2/2


In [3]:
# Create a test collection
TEST_COLLECTION_NAME = "RSPY_643_TEST_COLLECTION"
collection = create_test_collection(TEST_COLLECTION_NAME)

# Check the catalog for RSPY_643_TEST_COLLECTION
items = catalog_client.get_items(TEST_COLLECTION_NAME)
assert not list(items)

CADIP_SESSION_FILTER = "id=S1A_20200105072204051312" # Session id "platform='sentinel-1a'" "id=S1A_20200105072204051312" S3A_20250109134406046340
AUXIP_CQL2_FILTER = {
    "filter": {
        "op": "and",
        "args": [
            {
                "op": "=",
                "args": [
                    {
                        "property": "product:type"
                    },
                    "AX___OSF_AX"
                ]
            },
            {
                "op": "=",
                "args": [
                    {
                        "property": "published"
                    },
                    "2016-01-01T00:00:00.000Z/2016-12-31T23:59:59.999Z"
                ]
            }
        ]
    },
    "sortby": [
        {
            "field": "start_datetime",
            "direction": "desc"
        }
    ],
    "limit": 10
}


13:07:34.051 [INFO] (rs_client.rs_client) Retrieving all items from collection 'alex:RSPY_643_TEST_COLLECTION'.


In [4]:
# Other imports
import getpass
import os
import os.path as osp
from resources import prefect_utils

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    PREFECT_BLOCK_S3.bucket_name,
    PREFECT_BLOCK_S3.bucket_folder,
    "users",
    os.environ.get("RSPY_HOST_USER", getpass.getuser()),
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

# For each data: 
# input_config_dir: s3 bucket folder that contains the configuration files (NOT THE VOLUMINOUS DATA !).
# It will be downloaded locally.
# payload_file: input yaml configuration file to pass to the triggering. Local to the 'input_config_dir'.
# output_data_dir: s3 bucket directory that will contain the generated data.
    # "input_config_dir": s3_config,
    # "payload_file": "s3/s3_dordop_payload_S3A_20250109134406046340.yaml",
    # "output_data_dir": f"{s3_output}/s3",
    # "input_config_dir": s3_config,
    # "payload_file": "s1/iw_joborder.short.local.yaml",
    # "output_data_dir": f"{s3_output}/s1.short",
flow_parameters = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_l0_demo_payload_dpr_mockup_template.yaml",
    "output_data_dir": f"{s3_output}/s3",
    "owner_id": OWNER_ID,
    "collection_name": TEST_COLLECTION_NAME,
    "cadip_stac_filter": CADIP_SESSION_FILTER, 
    "auxip_cql2_filter": AUXIP_CQL2_FILTER,
    "adgs_station": "ADGS",
    "cadip_station": "CADIP",
    "staging_timeout": 120,
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

13:07:34.217 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'prefect-share' path 'sub/dir/users/alex/l0/config/logging_config.yaml'.

13:07:34.219 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_dordop_payload_S3A_20250109134406046340.yaml_backup' to the bucket 'prefect-share' path 'sub/dir/users/alex/l0/config/s3/s3_dordop_payload_S3A_20250109134406046340.yaml_backup'.

13:07:34.220 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_3A.yaml' to the bucket 'prefect-share' path 'sub/dir/users/alex/l0/config/s3/l0_processor_configuration_3A.yaml'.

13:07:34.220 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml' to the bucket 'prefect-share' path 'sub/dir/users/alex/l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml'.

13:07:34.221 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_l0_demo_payload_dpr_mockup.yaml_backup' to the bucket 'prefect-share' path 'sub/dir/users/alex/l0/config/s3/s3_l0_demo_payload_dpr_mockup.yaml_backup'.

...

13:07:34.224 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/.ipynb_checkpoints/s3_dordop_payload_S3A_20250109134406046340-checkpoint.yaml' to the bucket 'prefect-share' path 'sub/dir/users/alex/l0/config/s3/.ipynb_checkpoints/s3_dordop_payload_S3A_20250109134406046340-checkpoint.yaml'.

13:07:34.225 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/.ipynb_checkpoints/l0_processor_configuration_3A-checkpoint.yaml' to the bucket 'prefect-share' path 'sub/dir/users/alex/l0/config/s3/.ipynb_checkpoints/l0_processor_configuration_3A-checkpoint.yaml'.

13:07:34.226 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/alex/l0/config/s1/iw_configuration.yaml'.

13:07:34.227 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_joborder.short.local.yaml' to the bucket 'prefect-share' path 'sub/dir/users/alex/l0/config/s1/iw_joborder.short.local.yaml'.

13:07:34.273 | INFO    | prefect.S3Bucket - Uploaded 11 files from 'l0/config' to the bucket 'prefect-share' path 'sub/dir/users/alex/l0/config/s1/iw_joborder.short.local.yaml'

In [5]:
# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster_eopf.name
os.environ["DASK_CLUSTER_STAGING_NAME"] = dask_cluster_staging.name

if not local_mode:
    os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_ADDRESS"]

# Setup adaptive scaling
#dask_gateway.adapt_cluster(dask_cluster.name, minimum=1, maximum=scale)

In [6]:
read_apikey(save_to_env = False, overwrite = False)

TypeError: read_apikey() got an unexpected keyword argument 'save_to_env'

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [ ]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./s3l0_demo_processor.yaml"

In [ ]:
deploy_name = "S3L0-demo-processor/sprint22-S3L0-demo-processor"
await prefect_utils.wait_for_deployment(deploy_name)

## 3. Run Prefect flow

In [ ]:
output_data_dir = flow_parameters["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(flow_parameters) # flow parameters

In [ ]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

In [ ]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 6. Shutdown the dask clusters

In [ ]:
# You can scale the clusters to 0 workers
dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)
dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)

# Or shutdown the clusters
shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)

# Close the python objects
close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [ ]:
from importlib import reload
debug_flow = True

In [ ]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway_staging, None)
    shutdown_dask_clusters(dask_gateway_eopf, None)
    init_dask_cluster_eopf(scale=2)
    init_dask_cluster_staging(scale=2)

    from resources.dask_utils import *
    os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster_eopf.name
    os.environ["DASK_CLUSTER_STAGING_NAME"] = dask_cluster_staging.name

In [ ]:
if debug_flow:
    import s3l0_demo_processor
    reload(s3l0_demo_processor)
    results = s3l0_demo_processor.s3l0_demo_processor(**flow_parameters)
    display(results)

In [ ]:
result = catalog_client.remove_collection(collection_name)
assert result.json()["deleted collection"] == collection_name